# Uplift Modelling notes

This notebook includes all my notes and research for the necessary stats background to take up this project. It's the first part I am working on as I think it's the most important part of the project.

### Overview

The primary goal of uplift modelling is fundamentally to make causal inference. It aims to measure the true incremental effect of some intervention (like an ad, email, or medical treatment) by comparing what happens to a subject with the treatment versus without it (the control group). Traditional machine learning predicts correlation (e.g., "Will this customer buy a product?"), which often captures people who would have bought anyway. Uplift modeling uses causal techniques to answer the counterfactual question: "Will this customer buy only because they received the treatment?"

Uplift modelling is used to estimate incremental effect an action has on every individual (eg. how much more likely is someone to buy a product if they are shown an ad for it). It allows one to have targeted decisions like focusing on persuadable people, while avoiding people who'd convert no matter what or never convert. Often referred to as treatment-effect modelling or heterogeneous treatment-effect estimation.

Typically the workflow looks like: (1) define treatment, outcome, timing, and eligible population; (2) use a randomised experiment where possible, or adjust for measured confounding; (3) fit an uplift/CATE model using treated and control outcomes; (4) rank people by predicted uplift; and (5) evaluate the ranking on held-out data with Qini/AUUC and policy value. The hard part is that each person has only one observed outcome, so the counterfactual must be inferred from comparable treated and untreated people. Uplift is therefore causal, assumption-dependent, and different from predicting response alone.

The model will split the population into 4 groups based on outcomes:

    1. Persuadables: Respond only if they receive the treatment (target these).
    2. Sure Things: Respond regardless of whether they receive the treatment (save your budget).
    3. Lost Causes: Do not respond either way (ignore).
    4. Sleeping Dogs (or No Disturbs): React negatively or churn when treated (avoid).

### Outcomes and targets

We first define the Treatment (T) which represents the specific action, intervention, or stimulus applied to a subject. This allows us to measured the causal impact of the action compared to a control group that receives no intervention.

For each person then we define two potential outcomes: $Y(1)$ if treated and $Y(0)$ if untreated. If we look at everyone who responds after a treatment this will capture people who would have responded no matter what.

We use the Average Treatment Effect (ATE), $E[\tau(X)]$, to define a global baseline to know if an intervention works on average before personalising it. ATE is the overall, average difference in an outcome between a group that receives a treatment and a control group that does not. Relying solely on ATE assumes a treatment impacts everyone similarly which is not always the case. It is the expected impact if you applied the treatment to the entire population.

The Average Treatment Effect on the Treated (ATT), $E[\tau(X)\mid T=1]$, quantifies the average causal impact of an intervention specifically on the group of individuals who actually received that intervention. It is the expected impact restricted only to those who were actually treated.

In uplift modelling to find who to target next we use the conditional average treatment effect (CATE) which is the true incremental change caused by specifically the treatment itself defined as,

$$\tau(x) = E[Y(1)-Y(0)\mid X=x],$$

where Y(1) is the outcome if treatment is applied, Y(0) is outcome if treatment is withheld (control group), and $X$ is individual customer features or covariates (a continuous or measured variable that can influence the dependent variable). It is the individualised or personalised uplift score based on specific characteristics (e.g., age, tenure). 

Finally, we also have Individual Treatment Effect (ITE), $\tau_i$, which measures the exact causal impact of a specific intervention (such as a discount, coupon, or ad) on an individual customer i mathematically defined as the difference between two potential outcomes:

$$\tau_i = Y_i(1)-Y_i(0).$$

Because you can never observe both states simultaneously for the same person (you either treat them or you don't), the true ITE is unobservable and must be estimated using causal inference and machine learning.

### Typical Assumptions

To identify causal effects from treatment assignment

1. Stable Unit Treatment Value Assumption (SUTVA): One unit's treatment assignment does not affect another unit's potential outcomes, and there is only a single version of the treatment. Treatment assignment is how units (such as people, patients, or economic entities) are allocated to receive an intervention vs a comparison or control condition.

2. Ignorability / Unconfoundedness: Given observed pre-treatment covariates, the treatment assignment is independent of the potential outcomes. $(Y(0),Y(1)) \perp T\mid X$. In an experiment this follows from randomisation; in observational data it is an assumption.

3. Positivity/overlap: Every unit has a non-zero probability of receiving either treatment or control. $0 < e(X)=P(T=1\mid X) < 1$ for relevant $X$.


The propensity score, $e(x)=P(T=1\mid X=x)$, summarises treatment assignment. It is the conditional probability that a subject will receive a specific treatment or intervention, given a set of observed baseline characteristics (covariates). It was introduced as a technique to reduce selection bias and confounding in observational studies where random assignment is not possible. Hence, we fit a classifier for $T$ using pre-treatment features and inspect its cross validated Area Under the Curve (AUC) and predicted propensities. High AUC can mean the assignment mechanism is learnable (often confounding or targeting); extreme propensities mean weak overlap and unstable causal estimates. Low predictability is not proof of randomisation, and high predictability is not automatically bad: it can be legitimate targeting, but it demands careful overlap and sensitivity checks.

### Balance tables

Compare covariates between treated and control groups before and after weighting/matching. The standardised mean difference (SMD) for a continuous variable is

$$\mathrm{SMD}=\frac{\bar x_1-\bar x_0}{\sqrt{(s_1^2+s_0^2)/2}}.$$

For binary variables use the analogous difference in proportions divided by the pooled standard deviation. SMD is scale-free and does not become artificially significant just because $n$ is large. A common heuristic is $|\mathrm{SMD}|<0.1$ after adjustment, but inspect distributions too: balance on means can hide tail or nonlinear imbalance. A balance table is a design diagnostic, not evidence that the causal assumptions are true.

### Sources:

1. https://imai.fas.harvard.edu/teaching/files/causal.pdf
2. https://www.ibm.com/think/topics/causal-inference
3. https://arxiv.org/abs/2308.09066
4. https://medium.com/data-reply-it-datatech/uplift-modeling-predict-the-causal-effect-of-marketing-communications-24385fb04f2e
5. https://en.wikipedia.org/wiki/Propensity_score_matching
6. https://lost-stats.github.io/Presentation/Tables/Balance_Tables.html